# EX: Hidden Markov Models and Online Filtering

In Electronic Warfare (EW), an Electronic Support Measures (ESM) sensor pod intercepting radio frequency (RF) signals cannot directly view the operating mode of an adversary's multi-function radar. The radar's operational state—such as Search mode vs. Track mode—is a Hidden State ($X_t$).

The ESM receiver only detects discrete intercepted radar pulse bursts, such as Low PRF or High PRF, which constitute noisy Observations ($E_t$).

The objective of this exercise is to build an online Bayesian Filtering engine using the Forward Algorithm. The algorithm maintains and updates the AI agent's Belief State ($B(X_t) = P(X_t \mid e_{1:t})$) across a time-series of intercepted emissions by alternating between:

**Time Prediction:** Projecting the previous belief across the state transition model.

**Observation Update:** Conditioning on the new sensor intercept using Bayes' Rule.

## Lab Workflow & Steps Taken in the Code

**Step 1:** Define HMM System MatricesEstablish the discrete state space (Search, Track), observation space (Low, High), initial prior distribution vector $\mathbf{b}_0$, transition matrix $\mathbf{T}$, and emission likelihood matrix $\mathbf{O}$.

**Step 2:** Implement the Forward Filtering StepCreate a function forward_filter_step(belief, observation, T, O) that executes:

$$\bar{B}(X_t) = \sum_{x_{t-1}} P(X_t \mid x_{t-1}) B(x_{t-1}) \quad \implies \quad B(X_t) = \alpha P(e_t \mid X_t) \bar{B}(X_t)$$

**Step 3:** Process a Tactical Intercept SequenceSimulate an operational intercept scenario over three discrete time steps with evidence sequence:

$$\mathbf{e} = [E_1 = \text{High PRF}, \, E_2 = \text{High PRF}, \, E_3 = \text{Low PRF}]$$

**Step 4:** Analyze and Tabulate Belief EvolutionConstruct a Pandas DataFrame tracking the prior, predicted, and updated belief distributions across the mission timeline.

In [1]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Step 1: Define Model Parameters
# -------------------------------------------------------------------------
states = ['Search', 'Track']
observations = ['Low', 'High']

# Initial Prior Belief at t=0: P(Search) = 0.80, P(Track) = 0.20
b_0 = np.array([0.80, 0.20])

# Transition Matrix T[i, j] = P(X_t = j | X_{t-1} = i)
# Row 0: Search -> [Search, Track]
# Row 1: Track  -> [Search, Track]
T = np.array([
    [0.70, 0.30],
    [0.10, 0.90]
])

# Emission Matrix O[i, k] = P(E_t = k | X_t = i)
# Row 0: Search -> [Low, High]
# Row 1: Track  -> [Low, High]
O = {
    'Low':  np.array([0.80, 0.10]),
    'High': np.array([0.20, 0.90])
}

# -------------------------------------------------------------------------
# Step 2: Implement Single-Step Forward Filtering
# -------------------------------------------------------------------------
def forward_filter_step(current_belief, observation, transition_matrix, emission_dict):
    """
    Executes one cycle of HMM filtering:
    1. Prediction (Time Update): \bar{B}(X_t) = B(X_{t-1}) * T
    2. Measurement Update: B(X_t) = \alpha * EmissionLikelihood * \bar{B}(X_t)
    """
    # 1. Prediction step (Matrix-Vector multiplication)
    predicted_belief = np.dot(current_belief, transition_matrix)
    
    # 2. Measurement update step (Element-wise weighting by likelihood)
    likelihood = emission_dict[observation]
    unnormalized_belief = likelihood * predicted_belief
    
    # 3. Normalization constant
    marginal_evidence = np.sum(unnormalized_belief)
    updated_belief = unnormalized_belief / marginal_evidence
    
    return predicted_belief, updated_belief, marginal_evidence

# -------------------------------------------------------------------------
# Step 3: Run Filtering over Mission Intercept Stream
# -------------------------------------------------------------------------
mission_intercepts = ['High', 'High', 'Low']
current_b = b_0.copy()

history = [{
    'Time': 0,
    'Obs': 'Initial',
    'P_pred(Search)': np.nan,
    'P_pred(Track)': np.nan,
    'P_filt(Search)': current_b[0],
    'P_filt(Track)': current_b[1],
    'P(Evidence)': np.nan
}]

for t, obs in enumerate(mission_intercepts, start=1):
    pred_b, current_b, p_ev = forward_filter_step(current_b, obs, T, O)
    history.append({
        'Time': t,
        'Obs': obs,
        'P_pred(Search)': pred_b[0],
        'P_pred(Track)': pred_b[1],
        'P_filt(Search)': current_b[0],
        'P_filt(Track)': current_b[1],
        'P(Evidence)': p_ev
    })

# -------------------------------------------------------------------------
# Step 4: Display Results
# -------------------------------------------------------------------------
df_results = pd.DataFrame(history)
print("=" * 75)
print("TACTICAL HMM FILTERING ENGINE: RADAR MODE TRACKING")
print("=" * 75)
print(df_results.to_string(index=False))
print("=" * 75)

TACTICAL HMM FILTERING ENGINE: RADAR MODE TRACKING
 Time     Obs  P_pred(Search)  P_pred(Track)  P_filt(Search)  P_filt(Track)  P(Evidence)
    0 Initial             NaN            NaN        0.800000       0.200000          NaN
    1    High        0.580000       0.420000        0.234818       0.765182     0.494000
    2    High        0.240891       0.759109        0.065873       0.934127     0.731377
    3     Low        0.139524       0.860476        0.564683       0.435317     0.197667


## Interpreting the Results

* **Evidence Dominance Over Priors ($t=1$):** At deployment ($t=0$), the AI assumed an 80% prior probability of the radar being in Search mode. Intercepting a single High PRF pulse immediately swung the filtered belief to 76.52% Track. The diagnostic power of the sensor likelihood ($P(H \mid \text{Track}) = 0.90$ vs. $P(H \mid \text{Search}) = 0.20$) rapidly overcame the prior base rate.

* **Belief Consolidation ($t=2$):** When a second consecutive High PRF emission was received, our confidence that the radar is tracking surged to 93.41%. This demonstrates how persistent observations reinforce belief states in stochastic environments.

* **Mode Switching ($t=3$):** Intercepting a Low PRF pulse at $t=3$ immediately cut the Track belief from 93.41% down to 43.53%, restoring Search as the leading hypothesis (56.47%). Because radars switch modes dynamically, the HMM does not permanently lock onto Track mode, enabling autonomous electronic defense systems to detect when an adversary breaks lock.